In [107]:
# auto-reload changes to imported modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [108]:
import compile_tables as ct
import compile_file as cf
import parser_dat as p
import re
import pandas as pd
from missing_logk import MissingSolutionSpecies

Creating pandas dataframes for Solution Master Species (SMS)

In [109]:
db_list = p.phreeqc_database_list('../databases/test_data')
mst = ct.compile_master_solution_table(db_list)

Create ranking of databases to add from and assign them a priority

In [110]:
rank = {
    '#llnl.dat': 1,
    '#minteq.v4.dat': 2,
    '#phreeqc.dat': 3,
    '#Tipping_Hurley.dat': 4,
}
mst['priority'] = mst['source'].apply(lambda x: rank[x] if x in rank else 5)
mst = mst.sort_values(by=['priority'])

Create the result_mst from the llnl dat

In [111]:
result_mst = mst[mst['source'] == '#llnl.dat']
result_mst.head()

,element,species,alk,gfw_formula,element_gfw,source,priority
0,Acetate,HAcetate,0.0,Acetate,59.,#llnl.dat,1
145,Pd,Pd+2,0.0,Pd,106.42,#llnl.dat,1
147,Pm(+2),Pm+2,0.0,Pm,None,#llnl.dat,1
148,Pm(+3),Pm+3,0.0,Pm,None,#llnl.dat,1
149,Pr,Pr+3,0.0,Pr,140.9076,#llnl.dat,1


Seperate minteq SMS

In [112]:
minteq = mst[mst['source'] == '#minteq.v4.dat']
minteq.head()

,element,species,alk,gfw_formula,element_gfw,source,priority
102,Isopropylamine,Isopropylamine,1.0,59.111,59.111,#minteq.v4.dat,2
101,Propylamine,Propylamine,1.0,59.111,59.111,#minteq.v4.dat,2
100,Ethylenediamine,Ethylenediamine,2.0,60.099,60.099,#minteq.v4.dat,2
99,Hexylamine,Hexylamine,1.0,101.192,101.192,#minteq.v4.dat,2
98,Dimethylamine,Dimethylamine,1.0,45.084,45.084,#minteq.v4.dat,2


Create mask for elements present in minteq but missing from llnl

In [113]:
missing = ~minteq['element'].isin(result_mst['element'])

Filter minteq database to only elements missing from llnl

In [114]:
missing_species = minteq[missing]
print(missing_species.head())
result_mst['element'].isin(missing_species['element']).any()

             element          species  alk gfw_formula element_gfw  \
102   Isopropylamine   Isopropylamine  1.0      59.111      59.111   
101      Propylamine      Propylamine  1.0      59.111      59.111   
100  Ethylenediamine  Ethylenediamine  2.0      60.099      60.099   
99        Hexylamine       Hexylamine  1.0     101.192     101.192   
98     Dimethylamine    Dimethylamine  1.0      45.084      45.084   

             source  priority  
102  #minteq.v4.dat         2  
101  #minteq.v4.dat         2  
100  #minteq.v4.dat         2  
99   #minteq.v4.dat         2  
98   #minteq.v4.dat         2  


False

Hg and Sb(OH)6- mess with the database for some reason. Dropping them from the missing list here

In [115]:
print(missing_species[missing_species['species'].str.contains('Sb(OH)6-', regex=False)])
to_drop = ['Hg', 'Sb(OH)6-']
missing_species = missing_species[~missing_species['species'].isin(to_drop)]
print("Sb(OH)6- present in the df:",(missing_species['species'] == 'Sb(OH)6-' ).any())
missing_species.head()

   element   species  alk gfw_formula element_gfw          source  priority
68  Sb(+5)  Sb(OH)6-  0.0          Sb        None  #minteq.v4.dat         2
Sb(OH)6- present in the df: False


,element,species,alk,gfw_formula,element_gfw,source,priority
102,Isopropylamine,Isopropylamine,1.0,59.111,59.111,#minteq.v4.dat,2
101,Propylamine,Propylamine,1.0,59.111,59.111,#minteq.v4.dat,2
100,Ethylenediamine,Ethylenediamine,2.0,60.099,60.099,#minteq.v4.dat,2
99,Hexylamine,Hexylamine,1.0,101.192,101.192,#minteq.v4.dat,2
98,Dimethylamine,Dimethylamine,1.0,45.084,45.084,#minteq.v4.dat,2


See exactly what is not making the cut. Anything that has the same element won't be added to the new database.

In [116]:
mst[mst.duplicated(subset=['element'], keep='first')].head()

,element,species,alk,gfw_formula,element_gfw,source,priority
116,Acetate,Acetate-,1.0,59.045,59.045,#minteq.v4.dat,2
86,V,VO2+,-2.0,V,50.94,#minteq.v4.dat,2
80,Tl(+3),Tl(OH)3,0.0,Tl,None,#minteq.v4.dat,2
10,B,H3BO3,0.0,B,10.81,#minteq.v4.dat,2
76,Sn(+4),Sn(OH)6-2,0.0,Sn,None,#minteq.v4.dat,2


Create the result_sp (Solution Species) table from llnl

In [117]:
sp = ct.compile_solution_species_table(db_list)
result_sp = sp[sp['source'] == 'llnl.dat']
result_sp.head(5)

,equation,log_k,delta_h,gamma,add_logk,no_check,source
0,HAcetate = HAcetate,0.0,"(0, kJ/mol)",None,None,None,llnl.dat
1,Ag+ = Ag+,0.0,"(0, kJ/mol)",None,None,None,llnl.dat
2,Al+3 = Al+3,0.0,"(0, kJ/mol)",None,None,None,llnl.dat
3,Am+3 = Am+3,0.0,"(0, kJ/mol)",None,None,None,llnl.dat
4,Ar = Ar,0.0,"(0, kJ/mol)",None,None,None,llnl.dat


Using the apply function, we go through each species in the missing_species table and look for entires in the full sp (Solution Speices) table. If the missing species is present, we collect its index to a list. matched_series is built from the collected indexes. This will eventuall be concatenated to the result_sp dataframe.

In [118]:
import pandas as pd

# Sample data
series2 = sp['equation']

# Initialize an empty list to store all indexes
all_match_indexes = []

# Define a function to find indexes of matches and extend the all_match_indexes list
def find_and_collect_matches(entry):
    if entry == 'Sb(OH)6-':
        print('What the hell')
    match_indexes = series2[series2.str.contains(entry, regex=False)].index.tolist()
    if match_indexes:
        all_match_indexes.extend(match_indexes)

# Apply the function to series1
missing_species['species'].apply(find_and_collect_matches)

# Remove duplicates from the list of all match indexes
all_match_indexes = list(set(all_match_indexes))

# Index series2 using the list of match indexes
matched_series = sp.loc[all_match_indexes]
matched_series.head()

,equation,log_k,delta_h,gamma,add_logk,no_check,source
2048,Zn+2 + 2Diethylamine = Zn(Diethylamine)2+2,5.27,"(0, kJ)","(0, 0)",None,None,minteq.v4.dat
2049,Zn+2 + 3Diethylamine = Zn(Diethylamine)3+2,7.71,"(0, kJ)","(0, 0)",None,None,minteq.v4.dat
2050,Zn+2 + 4Diethylamine = Zn(Diethylamine)4+2,9.84,"(0, kJ)","(0, 0)",None,None,minteq.v4.dat
2051,Cd+2 + Diethylamine = Cd(Diethylamine)+2,2.73,"(0, kJ)","(0, 0)",None,None,minteq.v4.dat
2052,Cd+2 + 2Diethylamine = Cd(Diethylamine)2+2,4.86,"(0, kJ)","(0, 0)",None,None,minteq.v4.dat


We must drop equations with Hg(OH)2 and Sb(OH)6- manually. I don't know why but it does make the dat run :(

In [119]:
drop_re = re.compile('Hg[(]OH[)]2|Sb[(]OH[)]6-')
drop_index = matched_series[matched_series['equation'].str.contains(drop_re)].index
print(matched_series.shape)
matched_series = matched_series.drop(index=drop_index)
print(matched_series.shape)

(657, 7)
(631, 7)


In [120]:
result_mst = pd.concat([result_mst, missing_species], ignore_index=True)
result_mst['source'].unique()

array(['#llnl.dat', '#minteq.v4.dat'], dtype=object)

In [121]:
result_sp = pd.concat([result_sp, matched_series], ignore_index=True)
result_sp['source'].unique()

array(['llnl.dat', 'minteq.v4.dat'], dtype=object)

In [122]:
NAMED_EXPRESSIONS = """
NAMED_EXPRESSIONS
#
# formation of O2 from H2O
# 2H2O =  O2 + 4H+ + 4e-
#
    Log_K_O2
        log_k      -85.9951
        -delta_H	559.543	kJ/mol	# Calculated enthalpy of reaction	O2
#	Enthalpy of formation:	-2.9 kcal/mol
            -analytic   38.0229    7.99407E-03   -2.7655e+004  -1.4506e+001  199838.45
#	Range:  0-300

"""

In [123]:
with open('master_database.dat', 'w', encoding='utf-8') as f:
    f.write(NAMED_EXPRESSIONS)
    f.write("SOLUTION_MASTER_SPECIES\n#element\tmaster species\talkalinity\tgfw|formula\tgfw of element\tsource\n")
    result_mst.apply(lambda row: cf.write_mst(row, f), axis=1)
    f.write("\nSOLUTION_SPECIES\n")
    result_sp.apply(lambda row: cf.write_sp(row, f), axis=1)

print("File processing complete.")

File processing complete.
